# Coffee17 DCL-Style Local Learning V1

Matched five-fold decision experiment: raw-RGB MobileNetV3-Large + HBP + CE versus the identical inference core trained with DCL-style local region confusion, original/shuffled discrimination, and region alignment. Outer test is not accessed.


In [ ]:
NOTEBOOK_BUILD = "COFFEE17-DCL-LOCAL-V1"
SCIENTIFIC_CODE_COMMIT = "4dfacc01de28191c7bb11a00353a6bfc303adf5f"

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import importlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-dcl-local"
PROJECT = WORK / "coffee17-dcl-local-learning-v1-project"
OUT = PROJECT / "experiments/coffee17-dcl-local-learning-v1"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

RESUME_ZIP = WORK / "coffee17-dcl-local-learning-v1-resume-package.zip"

def write_resume_archive():
    if not PROJECT.exists():
        return None
    if RESUME_ZIP.exists():
        RESUME_ZIP.unlink()
    archive = Path(shutil.make_archive(
        str(RESUME_ZIP.with_suffix("")),
        "zip",
        root_dir=PROJECT,
    ))
    print("\nRESUME PACKAGE:", archive, flush=True)
    return archive

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    env = os.environ.copy()
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True, env=env)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command, cwd=cwd, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        write_resume_archive()
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

prior_projects = sorted(
    p for p in INPUT.rglob("coffee17-dcl-local-learning-v1-project")
    if p.is_dir()
)
resume_archives = sorted(
    p for p in INPUT.rglob("coffee17-dcl-local-learning-v1-resume-package.zip")
    if p.is_file()
)
if len(prior_projects) > 1 or len(resume_archives) > 1:
    raise RuntimeError("Resume source DCL ambigu: ditemukan lebih dari satu.")
if prior_projects and resume_archives:
    raise RuntimeError("Pilih satu resume source: project directory ATAU resume ZIP.")

if prior_projects:
    print("RESTORE PRIOR DCL PROJECT:", prior_projects[0])
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    shutil.copytree(prior_projects[0], PROJECT)
elif resume_archives:
    print("RESTORE DCL RESUME ZIP:", resume_archives[0])
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    PROJECT.mkdir(parents=True)
    shutil.unpack_archive(resume_archives[0], PROJECT)

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT / "analysis").mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    shutil.rmtree(REPO)
run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run(["git", "-C", REPO, "checkout", "--quiet", "--detach", SCIENTIFIC_CODE_COMMIT])

checked_out = subprocess.check_output(
    ["git", "-C", REPO, "rev-parse", "HEAD"], text=True
).strip()
if checked_out != SCIENTIFIC_CODE_COMMIT:
    raise RuntimeError(
        f"Checkout salah: {checked_out} != {SCIENTIFIC_CODE_COMMIT}"
    )
engine_text = (
    REPO / "src/bilinear_lmmd/engine/dcl_local_learning.py"
).read_text(encoding="utf-8")
for required_marker in (
    "def deterministic_adaptive_avg_pool2d",
    "def gpu_training_smoke_test",
    "optimizer_step_passed",
):
    if required_marker not in engine_text:
        raise RuntimeError(f"Scientific code marker hilang: {required_marker}")

run([sys.executable, "-m", "pip", "install", "-q", "-r",
     REPO / "requirements/preprocessing-study.txt"])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")

print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)
print("GPU:", torch.cuda.get_device_name(0))

run([
    sys.executable, "-m", "pytest", "-q",
    "tests/engine/test_dcl_local_learning.py",
    "tests/experiments/test_dcl_local_learning_runner.py",
], cwd=REPO)
print("PRE-TRAIN TESTS: PASS (includes strict CUDA DCL forward/backward/AdamW smoke)")

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)
from bilinear_lmmd.engine.physical_logit_residual_wr_hbp import configure_strict_determinism

det = configure_strict_determinism(42)
print("\n=== STRICT DETERMINISM ===")
print(json.dumps(det, indent=2))

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus Coffee17 original: "
        f"expected 979 images / 17 classes, observed {raw_count}/{len(by_class)}."
    )

ARCHIVE = WORK / "coffee17_dcl_original.zip"
PROV_LOCAL = WORK / "coffee17_dcl_provenance"
CANONICAL = WORK / "coffee17_dcl_original_v1"
FOLDS_LOCAL = WORK / "coffee17_dcl_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE, PROV_LOCAL, canonical_root=CANONICAL
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

CONFIG = REPO / "configs/dcl_local_learning/DCL_LOCAL_V1.yaml"

for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_dcl_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)

    materialize_preprocessing_development(
        CANONICAL,
        FOLDS_LOCAL / "clean_manifest.json",
        FOLDS_LOCAL / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_dcl_local_learning",
        "--config", CONFIG,
        "--fold", str(fold),
        "--data-root", DEV,
        "--output-root", OUT,
        "--required-commit", SCIENTIFIC_CODE_COMMIT,
        "--device", "cuda:0",
        "--resume",
        "--authorize-training",
    ], cwd=REPO, log_path=PROJECT / "logs" / f"fold_{fold}.log")

    result = json.loads(
        (OUT / f"fold_{fold}" / "seed42" / "pair_result.json").read_text()
    )
    d = result["DELTA_DCL_MINUS_CE"]
    print(
        f"FOLD {fold} | "
        f"CE Macro={result['HBP_CE']['macro_f1']:.4f} | "
        f"DCL Macro={result['HBP_DCL']['macro_f1']:.4f} | "
        f"Delta Macro={d['macro_f1']:+.4f} | "
        f"Delta Hard={d['hard_class_f1']:+.4f} | "
        f"Delta Worst={d['worst_class_f1']:+.4f} | "
        f"Rescue/Damage={result['paired_outcomes']['rescue']}/"
        f"{result['paired_outcomes']['damage']} | "
        f"PairDelta={result['targeted_confusion_delta']['TOTAL']:+d}"
    )

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

SUMMARY = PROJECT / "analysis" / "dcl_local_learning_summary.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_dcl_local_learning_summary",
    "--output-root", OUT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
d = summary["delta_dcl_minus_ce"]
gate = summary["screening_gate"]

print("\n=== DECISION ===")
print("Gate:", gate["decision"])
print(
    "Macro delta:",
    f"{d['macro_f1']['mean']:+.4%}",
    "| positive folds:", d["macro_f1"]["positive_folds"], "/5",
)
print(
    "Hard delta:",
    f"{d['hard_class_f1']['mean']:+.4%}",
    "| positive folds:", d["hard_class_f1"]["positive_folds"], "/5",
)
print("Worst delta:", f"{d['worst_class_f1']['mean']:+.4%}")
print(
    "Audited pair confusions:",
    gate["control_targeted_confusions_total"],
    "->",
    gate["dcl_targeted_confusions_total"],
)
print(
    "Rescue / damage:",
    summary["paired_prediction_outcomes"]["rescue"],
    "/",
    summary["paired_prediction_outcomes"]["damage"],
)
print("Outer test accessed:", summary["outer_test_accessed"])

PKG = WORK / "coffee17-dcl-local-learning-v1-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)
shutil.copy2(SUMMARY, PKG / SUMMARY.name)
shutil.copy2(CONFIG, PKG / CONFIG.name)
shutil.copy2(
    REPO / "docs/protocols/COFFEE17_DCL_LOCAL_LEARNING_V1.md",
    PKG / "COFFEE17_DCL_LOCAL_LEARNING_V1.md",
)

for fold in range(1, 6):
    src = OUT / f"fold_{fold}" / "seed42"
    dst = PKG / f"fold_{fold}"
    dst.mkdir(parents=True)
    shutil.copy2(src / "pair_result.json", dst / "pair_result.json")
    for arm in ("HBP_CE", "HBP_DCL"):
        arm_src = src / arm
        arm_dst = dst / arm
        arm_dst.mkdir(parents=True)
        for name in ("run_contract.json", "history.json", "run_config.yaml"):
            p = arm_src / name
            if p.is_file():
                shutil.copy2(p, arm_dst / name)
        if (arm_src / "validation").is_dir():
            shutil.copytree(arm_src / "validation", arm_dst / "validation")

zip_path = Path(shutil.make_archive(
    str(WORK / "coffee17-dcl-local-learning-v1-analysis-package"),
    "zip",
    root_dir=PKG,
))
print("\nREADY:", zip_path)
print("SIZE:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")

if RESUME_ZIP.exists():
    RESUME_ZIP.unlink()

for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
